##  **Instance Normalization (InstanceNorm)**

* **How it works**: Normalizes **each sample independently, each channel independently**, over the **spatial dimensions** (H, W). The batch is never mixed, and channels are never mixed.
* **Formula**: Same standardization as the other norms, but mean/var are computed **per sample, per channel** across spatial locations:

$$
\hat{x}_{n,c} = \frac{x_{n,c} - \mu_{n,c}}{\sqrt{\sigma^2_{n,c} + \epsilon}}
$$

where $\mu_{n,c}$ and $\sigma^2_{n,c}$ are taken over the $H\times W$ spatial positions of channel $c$ in sample $n$.

* **Best for**:
  * **Style transfer** and image generation (GANs). Removing per-image, per-channel contrast/style statistics makes the network insensitive to the contrast of the original content image — exactly what you want when restyling.
  * Independent of batch size (like LayerNorm and GroupNorm), so it works with batch size 1.

### **Where it sits among the normalizations**

All four norms standardize $\hat{x} = \dfrac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$ — they differ only in **which axes the mean/variance are computed over**. For a tensor of shape $(N, C, H, W)$:

| Norm | Statistics computed over | Mixes batch? | Mixes channels? |
|---|---|---|---|
| **BatchNorm** | $N, H, W$ (per channel) | Yes | No |
| **LayerNorm** | $C, H, W$ (per sample) | No | Yes |
| **GroupNorm** | $H, W$ + a group of channels (per sample) | No | within group |
| **InstanceNorm** | $H, W$ (per sample, per channel) | No | No |

InstanceNorm is the **extreme case of GroupNorm with one channel per group** (`num_groups = num_channels`). This is the same shared diagram concept used across these notebooks: only the normalization region inside the $(N, C, H, W)$ block changes.

### **`nn.InstanceNorm2d` example**

```python
import torch
import torch.nn as nn

# Input: N=2 samples, C=3 channels, 4x4 spatial
x = torch.randn(2, 3, 4, 4)

# InstanceNorm2d: normalizes each (sample, channel) over its H*W spatial values.
# By default affine=False and tracks no running stats (unlike BatchNorm).
inorm = nn.InstanceNorm2d(3, affine=False)  # first arg is num_features (= num channels)

out = inorm(x)

# Each (n, c) slice now has ~zero mean and unit variance over its spatial dims:
print(out[0, 0].mean(), out[0, 0].std(unbiased=False))  # ~0.0, ~1.0
print(out.shape)                                         # torch.Size([2, 3, 4, 4])
```

### **Notes**

* `affine=True` adds a learnable per-channel scale $\gamma$ and shift $\beta$ (like the other norms).
* Unlike `BatchNorm2d`, `InstanceNorm2d` by default does **not** track running statistics, so it behaves the **same in training and evaluation** — there is no train/eval discrepancy.
* Because statistics are per channel over only $H\times W$, InstanceNorm needs a reasonable spatial size to be meaningful (it is not used for $1\times1$ feature maps or fully connected layers).
* Variants such as **Adaptive Instance Normalization (AdaIN)** replace the learned $\gamma, \beta$ with statistics computed from a style image, which is the core operation in fast neural style transfer.

**References:** Ulyanov et al., *Instance Normalization: The Missing Ingredient for Fast Stylization*, 2016; Huang & Belongie, *Arbitrary Style Transfer with AdaIN*, 2017.